In [1]:
# El objetivo de la practica es trabajar sobre el merge, sobre el excel de telefonia 
# cargamos los primeros 3 meses (ene-mar) -
# tambien cargaremos la hoja franja edades. Franja_Edades
import pandas as pd

ruta_fichero = '/Users/rogerdefez/Documents/Cursos i Llibres/Curso Barcelona Activa/2026-04-29_Python3_Ciencia-de-datos-Python-Institute-PCAD-31-Ed2/Datos_originales/Datos Telefonia Separados Meses Comerciales URL.xlsx'
fras_enero = pd.read_excel(ruta_fichero,sheet_name="Facturación Enero 2020", header=2)
fras_febrero = pd.read_excel(ruta_fichero,sheet_name="Facturación Febrero 2020", header=0)
fras_marzo = pd.read_excel(ruta_fichero,sheet_name="Facturación Marzo 2020", header=1)
franjas = pd.read_excel(ruta_fichero,sheet_name="Franja_Edades", header=0)

#Concatenar los 3 meses
fras_trimestre = pd.concat([fras_enero,fras_febrero,fras_marzo],ignore_index=True)

#Eliminar las filas en blanco
fras_trimestre = fras_trimestre.dropna(how='all')

# Vamos a calcualr la edad + la decada, que ya la teníamos del otro ejercicio Del dia 2026-04-30 _ Practica 3.2

fras_trimestre['Dias'] = (pd.Timestamp('today') - fras_trimestre['Fecha Nacimiento']).dt.days
fras_trimestre['Edad'] = fras_trimestre['Dias'] // 365

# Vamos a calcular la decada

fras_trimestre['Decada'] = (fras_trimestre['Edad'].astype(str).str[0] + '0').astype('Int64')

# 1. VAmos a combinar (merge) con el Dataframe Franjas, solo se pueden combinar de 2 en 2, 
# no se pueden combinar listas de dataframes, si el nombre del campo es identico usaremos on=
# si el nombre del campo es diferente, usaremos left_on (df izquierda) y right_on
# parael dataframe de la derecha. PEro el tipo de dato debe ser el mismo para ambos campos de ambos dataframes.
# How = el tipo de comnbinaxion INNER 
# How = 'inner' los que coinciden de los 2 dataframes
# 

franjas.info()
resultado = pd.merge(fras_trimestre, franjas, on="Decada", how='inner')

# Muestrame el dataframe, ojo el campo clavequeda único, es decir no pasa el campo de la tabla de la derecha

resultado


<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Decada       8 non-null      int64
 1   Franja       8 non-null      str  
 2   Descripción  8 non-null      str  
dtypes: int64(1), str(2)
memory usage: 568.0 bytes


,Nombre,Servicios,Genero,Fecha Nacimiento,Localidad,Fecha factura,Importe factura,Satisfacción,IdComercial,Dias,Edad,Decada,Franja,Descripción
0,Cliente 1,Móvil,Masculino,1969-12-24,Madrid,2020-01-01,92.0,10.0,6.0,20586,56,50,Franja Senior,Entre 50-59 Años
1,Cliente 10,Voz IP,Femenino,1989-06-26,Baleares,2020-01-01,8.0,4.0,9.0,13462,36,30,Franja Middle,Entre 30-39 Años
2,Cliente 100,ADSL,Masculino,1989-09-23,Madrid,2020-01-01,49.0,7.0,3.0,13373,36,30,Franja Middle,Entre 30-39 Años
3,Cliente 1000,Móvil,Masculino,1950-10-16,Madrid,2020-01-01,25.0,7.0,7.0,27595,75,70,Franja Platino,Entre 70-79 Años
4,Cliente 1001,Voz IP,Masculino,1966-09-18,Castilla La Mancha,2020-01-01,40.0,4.0,3.0,21779,59,50,Franja Senior,Entre 50-59 Años
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6142,Cliente 2008,Fibra,Femenino,1992-04-13,Barcelona,2020-03-01,65.0,3.0,3.0,12440,34,30,Franja Middle,Entre 30-39 Años
6143,Cliente 2009,ADSL,Masculino,1963-06-29,Barcelona,2020-03-01,71.0,7.0,5.0,22956,62,60,Franja Gold,Entre 60-69 Años
6144,Cliente 2010,Móvil + Fijo,Masculino,1976-01-17,Barcelona,2020-03-01,49.0,2.0,7.0,18371,50,50,Franja Senior,Entre 50-59 Años
6145,Cliente 2011,Fibra,Femenino,1982-07-10,Barcelona,2020-03-01,53.0,8.0,2.0,16005,43,40,Franja Classic,Entre 40-49 Años


El merge con diferentes nombres
#1. df_merged = pd.merge(df1,df2, left_on = "ID_Cliente", right_on="Codigo_Cliente, how='inner')

#2. Combinar varios merge
df_merged1 = pd.merge(df1,df2, on = "ID_Cliente",  how='inner')
df_merged = pd.merge(df_merged1,df3, on = "ID_Producto how='inner')

#3. Optimizado (a partir del segundo merge, solo poner). Concatenar merge a partir del segundo merge solo poner un df
df_merged = pd.merge(df1,df2, on = "ID_Cliente",  how='inner').merge(df3, on="ID_Producto", how='inner')

#4. Solo taer un campo
#antes de combinar tienes que filtrar columnas
franjas = franjas[['Decada','Franja']]

In [2]:
# Como se trabaja el how= en el merge

df_clientes = pd.DataFrame({
    "id":[1,2,3,4],
    "nombre":['Marta', 'Juan', 'Monica', 'Pedro']
})

#Vamos a comprobar si funciona
print(df_clientes.head(3))

df_pedidos = pd.DataFrame({
    "id_cliente":[1,2,2,5],
    "producto":['Teclado','Mouse','Monitor','Impresora']
})

#Vamos a comprobar si funciona
print(df_pedidos.head(1))



   id  nombre
0   1   Marta
1   2    Juan
2   3  Monica
   id_cliente producto
0           1  Teclado


In [3]:
# El INNER JOIN  (Interseccion)  Solo los registros que coinciden en los d2 dataframes

pd.merge(df_clientes,df_pedidos, left_on="id", right_on= 'id_cliente', how='inner')

,id,nombre,id_cliente,producto
0,1,Marta,1,Teclado
1,2,Juan,2,Mouse
2,2,Juan,2,Monitor


In [4]:
# El LEFT JOIN   (Interseccion) Todos los registros de la izquierdo y los que coincidan con la derecha

pd.merge(df_clientes,df_pedidos, left_on="id", right_on= 'id_cliente', how='left')

,id,nombre,id_cliente,producto
0,1,Marta,1.0,Teclado
1,2,Juan,2.0,Mouse
2,2,Juan,2.0,Monitor
3,3,Monica,NaN,NaN
4,4,Pedro,NaN,NaN


In [5]:
# El RIGHT JOIN   (Interseccion) Todos los registros de la derecha y los que coincidan con la izquierda

pd.merge(df_clientes,df_pedidos, left_on="id", right_on= 'id_cliente', how='right')

,id,nombre,id_cliente,producto
0,1.0,Marta,1,Teclado
1,2.0,Juan,2,Mouse
2,2.0,Juan,2,Monitor
3,NaN,NaN,5,Impresora


In [6]:
# El OUTER JOIN   (Interseccion) Todos los registros de los 2 dataframes

pd.merge(df_clientes,df_pedidos, left_on="id", right_on= 'id_cliente', how='outer')

,id,nombre,id_cliente,producto
0,1.0,Marta,1.0,Teclado
1,2.0,Juan,2.0,Mouse
2,2.0,Juan,2.0,Monitor
3,3.0,Monica,NaN,NaN
4,4.0,Pedro,NaN,NaN
5,NaN,NaN,5.0,Impresora


In [8]:
# El LEFT ANTI JOIN   (Interseccion) Todos los registros de los 2 dataframes

pd.merge(df_clientes,df_pedidos, left_on="id", right_on= 'id_cliente', how=)

SyntaxError: expected argument value expression (3880627812.py, line 3)

PRACTICA 5:   PArtiendo del fichero excel "datos telefonia-......"  Necesitamos averiguar la suma de importe factura por nombre de comercial y descripcion de la satisfacción, de las facturas de Enero a junio del 2020.

Practica 5.1: Y ademas ver el promedio de importe factura por zona de localidad(solo para valientes)

In [9]:

# LLamar libreria pandas
import pandas as pd
# 1. Ajustar la ruta del archivo
ruta_fichero = "/Users/rogerdefez/Documents/Cursos i Llibres/Curso Barcelona Activa/2026-04-29_Python3_Ciencia-de-datos-Python-Institute-PCAD-31-Ed2/Datos_originales/Datos Telefonia Separados Meses Comerciales URL.xlsx"

# 2. Descargar los dataframes de ene-jun + satisfaccion + comerciales
fras_enero = pd.read_excel(ruta_fichero,sheet_name="Facturación Enero 2020", header=2)

fras_febrero = pd.read_excel(ruta_fichero,sheet_name="Facturación Febrero 2020")

fras_marzo = pd.read_excel(ruta_fichero,sheet_name="Facturación Marzo 2020",header=1)

fras_abril = pd.read_excel(ruta_fichero,sheet_name="Facturación Abril 2020",header=1)

fras_mayo = pd.read_excel(ruta_fichero,sheet_name="Facturación Mayo 2020",header=1)

fras_junio = pd.read_excel(ruta_fichero,sheet_name="Facturación Junio 2020",header=2)

# 3. Concatenar los meses
fras_semestre = pd.concat([fras_enero, fras_febrero, fras_marzo, fras_abril, fras_mayo, fras_junio], ignore_index=True)

# 4. Descargamos los otros 2 dataframes
descripciones = pd.read_excel(ruta_fichero,sheet_name="Satisfacción Cliente")

comerciales = pd.read_excel(ruta_fichero,sheet_name="Comerciales")

# 5. Comprobar cargas

#fras_semestre.info()

#descripciones.info()

#comerciales.info()

# 6. Combinar - merge

resultado = pd.merge(fras_semestre, comerciales, on='IdComercial', how='inner').merge(descripciones, left_on='Satisfacción', right_on='Satisfacción Cliente', how='inner')

# 7. Cambiar el nombre de una columna

resultado = resultado.rename(columns={"Nombre_y":"Comercial"})

# 8. REalizamos la agrupacion en pantalla, con reset_index() una serie se convierte en un dataframe

resultado.groupby(['Comercial', 'Descripción_y'])['Importe factura'].sum().reset_index()  

# 9. Crear una referencia al dataframe

agrupacion = resultado.groupby(['Comercial', 'Descripción_y'])['Importe factura'].sum().reset_index()

agrupacion.to_csv('/Users/rogerdefez/Documents/Cursos i Llibres/Curso Barcelona Activa/2026-04-29_Python3_Ciencia-de-datos-Python-Institute-PCAD-31-Ed2/agrupacion.csv')


# Solucion 5.1
# Practica 5.1: Y ademas ver el promedio de importe factura por zona de localidad(solo para valientes)

localidades = pd.read_excel(ruta_fichero,sheet_name="Localidad", header=0)
localidades
#resultado

,Localidad,Población,Zona,Satisfacción
0,1# Madrid,3207000,Centro,5
1,2# Galicia,2766000,Norte,2
2,3# Valencia,792303,Este,4
3,4# Andalucía,8440000,Sur,7
4,5# Castilla y León,2520000,Centro,8
5,6# Baleares,1112000,Este,3
6,7# Asturias,1068000,Norte,6
7,8# Barcelona,1612000,Este,9
8,9# Bilbao,349356,Norte,2
9,10# Santander,177123,Norte,1


In [10]:
# Como obtener la localidad solo, con el metodo split (de la clase str)

# Dividir la columna por un caracteer, quedarnos con la segunda columna, indice 1
# Quiltar los espacios en blanco

localidades['Localidad'] = localidades['Localidad'].str.split("#").str[1].str.strip()
localidades

,Localidad,Población,Zona,Satisfacción
0,Madrid,3207000,Centro,5
1,Galicia,2766000,Norte,2
2,Valencia,792303,Este,4
3,Andalucía,8440000,Sur,7
4,Castilla y León,2520000,Centro,8
5,Baleares,1112000,Este,3
6,Asturias,1068000,Norte,6
7,Barcelona,1612000,Este,9
8,Bilbao,349356,Norte,2
9,Santander,177123,Norte,1


In [11]:
# Dejar solo 2 columnas.  (no necesitamos mas)

localidades = localidades[['Localidad', 'Zona']]
localidades

,Localidad,Zona
0,Madrid,Centro
1,Galicia,Norte
2,Valencia,Este
3,Andalucía,Sur
4,Castilla y León,Centro
5,Baleares,Este
6,Asturias,Norte
7,Barcelona,Este
8,Bilbao,Norte
9,Santander,Norte


In [ ]:
# Combinamos con el dataframe de resultado que lo tiene todo, pero no hago consulta nueva, lo hago sobre uno de los df

resultado = resultado.merge(localidades, on='Localidad', how='inner')

resultado.groupby('Zona')['Importe factura'].mean().round(2)


Zona
Centro    50.91
Este      49.93
Norte     49.73
Sur       48.84
Name: Importe factura, dtype: float64

Conectarnos con la libreria pdfplumber a un pdf y obtener sus tablas , que cargaremos en uno o varios dataframes
- Primero instalar pdfplumber
- >pip install pdfplumber


In [21]:
# Cargar las tablas de un pdf
# 0. Importar las librerias

import pdfplumber
import pandas as pd

# 1. Ajustar la ruta del fichero

ruta_pdf = '/Users/rogerdefez/Documents/Cursos i Llibres/Curso Barcelona Activa/2026-04-29_Python3_Ciencia-de-datos-Python-Institute-PCAD-31-Ed2/Datos_originales/Franjas edades.pdf'

# 2. Crear una lista para almacenar las tablas extraidas

tablas =[]

# 3. Vamos a crear un bucle para ir cargando las tablas, miraremos las diferentes tablas
#  de las diferentes paginas

with pdfplumber.open(ruta_pdf) as pdf:
    for page in pdf.pages:          # con esto indicamos que recorra todas las paginas del pdf
        # Extraer las tablas de la pagina que estoy recorriendo
        tablas_extraidas = page.extract_tables()
        
        # Vamos a recorrer la coleccion de tablas extraibles 
        for tabla in tablas_extraidas:
            df = pd.DataFrame(tabla)
            tablas.append(df)

# 4. Acceder al dataframe resultante

len(tablas) # Para saber cuantas tablas hay en la lista

# Lo mas optimo, es clonar el dataframe en un objeto para tenerlo separado 
franjas = tablas[0]

# Hacer la primera fila como encabezado
franjas = franjas.rename(columns=franjas.iloc[0])


# Y despues eliminamos la fila 0, con axis = 0 elimino filas, con axis = 1 elimino columnas

franjas = franjas.drop([0], axis=0).reset_index(drop=True)
franjas

,Decada,Franja,Descripción
0,10,Franja Kids,Entre 10 -19 Años
1,20,Franja Youth,Entre 20-29 Años
2,30,Franja Middle,Entre 30-39 Años
3,40,Franja Classic,Entre 40-49 Años
4,50,Franja Senior,Entre 50-59 Años
5,60,Franja Gold,Entre 60-69 Años
6,70,Franja Platino,Entre 70-79 Años
7,80,Franja White,Igual o mayor a 80 Años
